In [ ]:
from collections import defaultdict
from pathlib import Path

from bench_mhc.utils.io import save_json
from bench_mhc.utils.mhc import format_allele_name

# MHC1

> **Prerequisite:** the `!gsutil` calls below shell out to the Jupyter kernel's
> environment. If you run this notebook from your local `.venv` and see
> `gsutil: not found`, it's because Jupyter kernels do not source `.zshrc`.
> Add the gcloud SDK to `.zshenv` instead:
>
> ```sh
> echo 'export PATH="$HOME/google-cloud-sdk/bin:$PATH"' >> ~/.zshenv
> ```
>
> When running from the container (`make notebook`), `gsutil` is preinstalled
> in the image and no extra setup is needed.

In [ ]:
!gsutil cp -r "gs://bench-mhc/NetMHCpan-4.1/Original_data/NetMHCpan_train/" "../data/"

In [ ]:
# Print all files in NetMHCpan-4.1 dump

netmhcpan_dump_path = Path("../data/NetMHCpan_train/")
for file_path in netmhcpan_dump_path.iterdir():
    print(file_path)

In [ ]:
# Get number of samples per allele in original data

alleles_in_data = defaultdict(int)

for split in range(5):
    for type_ in ["el", "ba"]:
        with open(netmhcpan_dump_path / f"c00{split}_{type_}") as f:
            for line in f:
                peptide, target, allele = line.strip().split()
                alleles_in_data[allele] += 1

# Example
alleles_in_data["HLA-C14:02"]

In [ ]:
# Load the original mapping (MHC1)

mhc1_mapping = {}
with open(netmhcpan_dump_path / "MHC_pseudo.dat") as f:
    for line in f:
        name, sequence = line.strip().split()
        mhc1_mapping[name] = sequence

In [ ]:
example_allele = next(iter(mhc1_mapping))
print(example_allele, mhc1_mapping[example_allele])

In [ ]:
# Try to parse allele names and check whether 2 alleles with different pseudosequences
# have same compact representation

mhc1_original_alleles = set()
mhc1_new_alleles = {}
for allele, pseudoseq in mhc1_mapping.items():
    new_allele = format_allele_name(allele)
    if new_allele in mhc1_new_alleles and mhc1_new_alleles[new_allele] != pseudoseq:
        print(
            allele,
            "already in new allele with pseudoseq",
            mhc1_new_alleles[new_allele],
            "instead of",
            pseudoseq,
        )

    mhc1_original_alleles.add(allele)
    mhc1_new_alleles[new_allele] = pseudoseq

In [ ]:
print(
    f"MHCI alleles in unformatted (original) mapping: {len(mhc1_original_alleles):,} |"
    f"MHCI alleles in formatted mapping: {len(mhc1_new_alleles):,}"
)

In [ ]:
# Investigate two parsed alleles that have same compact representation to an allele
# with different pseudoseq

compact2allele2pseudoseq = defaultdict(dict)
for allele in [
    "Mamu-A1011:01",
    "Mamu-A11",
    "Mamu-A1:01101",
    "Mamu-A1:01104",
    "Mamu-A01",
    "Mamu-A1*00101",
    "Mamu-A1:00101",
    "Mamu-A1*01101",
    "Mamu-A1001:01",
    "Mamu-A1:01001",
]:
    new_allele = format_allele_name(allele)
    print(
        "Pseudoseq:",
        f"{mhc1_mapping[allele]:<40}" "Allele:",
        f"{allele:<20}",
        "Compact:",
        new_allele,
    )

    compact2allele2pseudoseq[new_allele][allele] = mhc1_mapping[allele]

In [ ]:
# Check allele that have common compact string but different pseudoseq

for compact_string, allele2pseudoseq in compact2allele2pseudoseq.items():
    print("\n---------------\nCompact:", compact_string)
    for allele, pseudoseq in allele2pseudoseq.items():
        print(f"{allele:<20}", pseudoseq, f"{alleles_in_data[allele]} samples")

### Summary

- `amani.1` and `gb1.7` cannot be `mhcgnomes`-parsed, so will be let like they are in the mapping.
- For alleles that share common compact string with different pseudo-sequences, we select the pseudoseq representation of the allele that has samples in the data, i.e.:
  - `mappings["Mamu__A101101"] = "YYSMYREKMTETYGNTLYITYEYYTWAVWAYEWY"`
  - `mappings["Mamu__A100101"] = "YYAMYRENMTENAVNTLYLRVEYYTWAVMAYQWY"`

In [ ]:
final_mhc1_mapping = {
    format_allele_name(allele): mhc1_mapping[allele]
    for allele in ["Mamu-A1*01101", "Mamu-A1*00101"]
}

for allele, pseudoseq in mhc1_mapping.items():
    new_allele = format_allele_name(allele)
    if new_allele in final_mhc1_mapping and final_mhc1_mapping[new_allele] != pseudoseq:
        print(
            allele,
            "already in mapping with pseudoseq",
            final_mhc1_mapping[new_allele],
            "instead of",
            pseudoseq,
        )
    else:
        final_mhc1_mapping[new_allele] = pseudoseq

In [ ]:
final_mhc1_mapping["Mamu__A101101"], final_mhc1_mapping["Mamu__A100101"]

In [ ]:
# Double-check it's the same as in original mapping

mhc1_mapping["Mamu-A1*01101"], mhc1_mapping["Mamu-A1*00101"]

## MHC2

In [ ]:
!gsutil cp -r "gs://bench-mhc/NetMHCpan-4.0/Original_data/NetMHCIIpan-4.0_train" "../data/"

In [ ]:
# Print all files in NetMHCIIpan-4.0 dump

netmhc2pan_dump_path = Path("../data/NetMHCIIpan-4.0_train")
for file_path in netmhc2pan_dump_path.iterdir():
    print(file_path)

In [ ]:
# Load the original mapping (MHC2)

mhc2_mapping = {}
with open(netmhc2pan_dump_path / "pseudosequence.2016.all.X.dat") as f:
    for line in f:
        name, sequence = line.strip().split()
        mhc2_mapping[name] = sequence

allele = next(iter(mhc2_mapping))
print(allele, mhc2_mapping[allele])

In [ ]:
# Try to parse allele names and check whether 2 alleles with different pseudosequences
# have same compact representation

mhc2_original_alleles = set()
mhc2_new_alleles = {}
for allele, pseudoseq in mhc2_mapping.items():
    new_allele = format_allele_name(allele)
    if new_allele in mhc2_new_alleles and mhc2_new_alleles[new_allele] != pseudoseq:
        print(
            allele,
            "already in new allele with pseudoseq",
            mhc2_new_alleles[new_allele],
            "instead of",
            pseudoseq,
        )

    mhc2_original_alleles.add(allele)
    mhc2_new_alleles[new_allele] = pseudoseq

In [ ]:
print(
    f"MHC2 alleles in unformatted (original) mapping: {len(mhc2_original_alleles):,} | "
    f"MHC2 alleles in formatted mapping: {len(mhc2_new_alleles):,}"
)

### Summary

- No issues in MHC2: one-one mapping between allele name and formatted one

In [ ]:
final_mhc2_mapping = {}

for allele, pseudoseq in mhc2_mapping.items():
    new_allele = format_allele_name(allele)
    if new_allele in final_mhc2_mapping and final_mhc2_mapping[new_allele] != pseudoseq:
        print(
            allele,
            "already in mapping with pseudoseq",
            final_mhc2_mapping[new_allele],
            "instead of",
            pseudoseq,
        )
    else:
        final_mhc2_mapping[new_allele] = pseudoseq

In [ ]:
final_mapping = {**final_mhc1_mapping, **final_mhc2_mapping}
print(
    f"Final mapping contains {len(final_mapping):,} alleles. (MHC1: {len(final_mhc1_mapping):,} ; "
    f"MHC2: {len(final_mhc2_mapping):,})"
)

In [ ]:
save_json(
    dict(sorted(final_mapping.items())),
    "../data/mappings/allele2netmhc_pseudo_seq.json",
    indent=2,
)